# Code to alter/ajust the orinal images to a set of images for the figure-ground pipeline

In [1]:
from PIL import Image, ImageOps
import os
import glob
from os import listdir
from os.path import isfile, join
import numpy as np
import cv2

In [2]:

# Empty the variations directory
files = glob.glob('images_variations/*')
for f in files:
    os.remove(f)

# Get the names of the original images
original_images = [f for f in listdir('images_original') if isfile(join('images_original', f))]
print(original_images)

['rubins_vase_color.jpg', 'rubins_vase_blackwhite.jpg', 'faces_vase_color.jpg']


In [3]:
for orininal_image in original_images:
    image_split = str.split(orininal_image, '.')
    image_name = image_split[0]
    image_extension = image_split[1]
    for border_size in range(0,45):
        border_size = border_size * 2
        image_border = Image.open(join('images_original',orininal_image))
        image_border = ImageOps.expand(image_border, border_size, 'black')
        image_border.save(join('images_variations', image_name + '_border_' + str(border_size) + '.' + image_extension))
    for angle in range(0,45):
        image_rotation = Image.open(join('images_original',orininal_image))
        image_rotation = image_rotation.rotate(angle, expand=True)
        image_rotation.save(join('images_variations',image_name + '_rotate_' + str(angle) + '.' + image_extension))
    for noise_probability in [0.2,0.5,0.6,0.7,0.8,0.85,0.9,0.95]:
        # Salt Pepper Noise
        img = np.copy(np.array(Image.open(join('images_original',orininal_image))))
        prob = noise_probability # probability of generating a noise pixel
        output = img.copy()
        if len(img.shape) == 2:
            black = 0
            white = 255            
        else:
            colorspace = img.shape[2]
            if colorspace == 3:  # RGB
                black = np.array([0, 0, 0], dtype='uint8')
                white = np.array([255, 255, 255], dtype='uint8')
            else:  # RGBA
                black = np.array([0, 0, 0, 255], dtype='uint8')
                white = np.array([255, 255, 255, 255], dtype='uint8')
        probs = np.random.random(output.shape[:2])
        output[probs < (prob / 2)] = black
        output[probs > 1 - (prob / 2)] = white
        image = Image.fromarray(output)
        image.save(join('images_variations', image_name + '_spnoise_' + str(int(noise_probability*100)) + '.' + image_extension))


In [13]:
# Create Gaussian Noise

img = np.copy(np.array(Image.open('faces_vase_color.jpg')))
mean = 0
stddev = 1
gamma = 1
gauss_noise = np.zeros(img.shape[:2])
cv2.randn(gauss_noise, mean, stddev)
gauss_noise = (gauss_noise*gamma).astype(np.uint8)

if len(img.shape) == 2:
    output = cv2.add(img, gauss_noise)
elif len(img.shape) == 3:
    merged = cv2.merge([gauss_noise, gauss_noise, gauss_noise])
    output = cv2.add(img, merged)


image = Image.fromarray(output)
image.save('gauss_noise.jpg')

In [16]:
# Salt Pepper Noise

img = np.copy(np.array(Image.open('faces_vase_color.jpg')))
prob = 0.8 # probability of generating a black pixel instead of a white one
output = img.copy()
if len(img.shape) == 2:
    black = 0
    white = 255            
else:
    colorspace = img.shape[2]
    if colorspace == 3:  # RGB
        black = np.array([0, 0, 0], dtype='uint8')
        white = np.array([255, 255, 255], dtype='uint8')
    else:  # RGBA
        black = np.array([0, 0, 0, 255], dtype='uint8')
        white = np.array([255, 255, 255, 255], dtype='uint8')
probs = np.random.random(output.shape[:2])
output[probs < (prob / 2)] = black
output[probs > 1 - (prob / 2)] = white

image = Image.fromarray(output)
image.save('saltpepper_noise.jpg')



In [9]:
amount = 0.004

image = Image.open('faces_vase_color.jpg')
output = np.copy(np.array(image))

# add salt
nb_salt = np.ceil(amount * output.size * 0.5)
coords = [np.random.randint(0, i - 1, int(nb_salt)) for i in output.shape]
output[coords] = 1

# add pepper
nb_pepper = np.ceil(amount* output.size * 0.5)
coords = [np.random.randint(0, i - 1, int(nb_pepper)) for i in output.shape]
output[coords] = 0

image = Image.fromarray(output)
image.save('salt_pepper_vase.jpg')


In [3]:
variations_images = [f for f in listdir('images_variations') if isfile(join('images_variations', f))]

print(variations_images)

['faces_vase_color_rotate_33.jpg', 'faces_vase_color_rotate_27.jpg', 'faces_vase_color_border_12.jpg', 'faces_vase_color_rotate_3.jpg', 'faces_vase_color_rotate_2.jpg', 'faces_vase_color_border_13.jpg', 'faces_vase_color_rotate_26.jpg', 'faces_vase_color_rotate_32.jpg', 'faces_vase_color_rotate_18.jpg', 'faces_vase_color_rotate_24.jpg', 'faces_vase_color_rotate_30.jpg', 'faces_vase_color_border_39.jpg', 'faces_vase_color_border_11.jpg', 'faces_vase_color_rotate_1.jpg', 'faces_vase_color_border_10.jpg', 'faces_vase_color_border_38.jpg', 'faces_vase_color_rotate_31.jpg', 'faces_vase_color_rotate_25.jpg', 'faces_vase_color_rotate_19.jpg', 'faces_vase_color_rotate_21.jpg', 'faces_vase_color_rotate_35.jpg', 'faces_vase_color_border_14.jpg', 'faces_vase_color_border_28.jpg', 'faces_vase_color_rotate_5.jpg', 'faces_vase_color_rotate_4.jpg', 'faces_vase_color_border_29.jpg', 'faces_vase_color_border_15.jpg', 'faces_vase_color_rotate_34.jpg', 'faces_vase_color_rotate_20.jpg', 'faces_vase_color_